<a href="https://colab.research.google.com/github/jiyanarikan/Computer-Vision-Covariate-Shift-/blob/main/CompVision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install wilds torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 5.5 MB/s eta 0:00:00


In [2]:
from wilds import get_dataset

In [3]:
print("Downloading dataset")

# This downloads to a folder in Colab env
dataset = get_dataset(dataset="poverty", download=True)
train_data = dataset.get_subset("train")
print(f"Total training images loaded: {len(train_data)}")

You can also download the dataset manually at https://wilds.stanford.edu/downloads.


13091954688Byte [07:57, 27441545.79Byte/s]                               


Extracting data/poverty_v1.1/archive.tar.gz to data/poverty_v1.1

It took 10.38 minutes to download and uncompress the dataset.

Total training images loaded: 9797


In [5]:
import torch
import torchvision.models as models
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm

In [6]:
# Load ResNet trained on standard images
weights = models.ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=weights)

# Leaves us with a model that outputs a flat vector of numbers
feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
feature_extractor.eval()

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 135MB/s]


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Con

In [7]:
from google.colab import drive
import os

drive.mount('/content/drive')

data_path = '/content/drive/MyDrive/CompVision_Poverty_Research'
if not os.path.exists(data_path):
    os.makedirs(data_path)

print(f"Data saved to: {data_path}")

Mounted at /content/drive
Data saved to: /content/drive/MyDrive/CompVision_Poverty_Research


In [8]:
# Print the field names without loading the whole array into a DF
print("Fields in metadata:", dataset.metadata_fields)

# Pull the metadata for the first 5 samples manually
for i in range(5):
    _, _, meta = train_data[i]
    print(f"Sample {i} metadata: {meta.tolist()}")

Fields in metadata: ['urban', 'y', 'country', 'from_source_domain']
Sample 0 metadata: [0.0, -0.959787368774414, 3.0, 1.0]
Sample 1 metadata: [0.0, -0.859114408493042, 3.0, 1.0]
Sample 2 metadata: [1.0, -0.3560349941253662, 3.0, 1.0]
Sample 3 metadata: [1.0, 1.0717921257019043, 3.0, 1.0]
Sample 4 metadata: [1.0, 0.874895453453064, 3.0, 1.0]


In [9]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
feature_extractor.to(device)

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Con

In [10]:
# Filter for 500 Urban (Source) and 500 Rural (Target) samples
metadata = train_data.metadata_array
urban_indices = torch.where(metadata[:, 0] == 1)[0][:500].tolist()
rural_indices = torch.where(metadata[:, 0] == 0)[0][:500].tolist()
subset_indices = urban_indices + rural_indices

In [11]:
loader = DataLoader(torch.utils.data.Subset(train_data, subset_indices), batch_size=32)

all_features = []
all_domains = [] # 1 = Urban, 0 = Rural
all_wealth = []  # Dependent variable (y)

In [12]:
with torch.no_grad():
    for imgs, labels, metadata in tqdm(loader):
        imgs = imgs[:, :3, :, :].to(device)
        feats = feature_extractor(imgs).view(imgs.size(0), -1)

        all_features.append(feats.cpu().numpy())
        all_domains.append(metadata[:, 0].numpy())
        all_wealth.append(labels.numpy())

100%|██████████| 32/32 [00:16<00:00,  1.97it/s]


In [13]:
X = np.vstack(all_features)
y_domain = np.concatenate(all_domains)
y_wealth = np.concatenate(all_wealth)

print("\nSuccess! X shape:", X.shape, "| y_domain shape:", y_domain.shape)


Success! X shape: (1000, 512) | y_domain shape: (1000,)


In [17]:
from sklearn.linear_model import Ridge
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. Split our data into Urban (Source) and Rural (Target)
X_urban = X[y_domain == 1]
y_urban = y_wealth[y_domain == 1]

X_rural = X[y_domain == 0]
y_rural = y_wealth[y_domain == 0]

In [18]:
# 2. Train the Naive Model (Urban only, no weights)
naive_model = Ridge(alpha=1.0)
naive_model.fit(X_urban, y_urban)

Ridge()

In [19]:
# 1. Train the classifier: Is this image Urban (1) or Rural (0)?
# This model learns the visual difference between the two environments.
clf = LogisticRegression(max_iter=1000)
clf.fit(X, y_domain)

# 2. Get the Propensity Scores (Probability of being Rural)
# probs[:, 0] is the probability the image belongs to the target (Rural) domain.
probs = clf.predict_proba(X)
p_rural = probs[:, 0]

# 3. Calculate the IPW Weights for the Urban images
# Formula: Weight = P(Target) / P(Source)
# This mathematically "re-balances" the Urban data to look like Rural data.
weights_all = p_rural / (1 - p_rural + 1e-6)

# 4. Extract only the weights for the Urban samples (where y_domain == 1)
urban_weights = weights_all[y_domain == 1]

print(f"Weights successfully recalculated!")
print(f"Number of urban weights: {len(urban_weights)}")

Weights successfully recalculated!
Number of urban weights: 500


In [20]:
# 3. Train the IPW Model (Urban only, WITH our calculated weights)
# Note: 'urban_weights' came from our Logistic Regression step earlier
ipw_model = Ridge(alpha=1.0)
ipw_model.fit(X_urban, y_urban, sample_weight=urban_weights)

Ridge()

In [21]:
# 4. Evaluate both on the Rural (Target) data
y_pred_naive = naive_model.predict(X_rural)
y_pred_ipw = ipw_model.predict(X_rural)

In [22]:
# 5. Calculate Accuracy (R-squared and MSE)
mse_naive = mean_squared_error(y_rural, y_pred_naive)
mse_ipw = mean_squared_error(y_rural, y_pred_ipw)

In [23]:
print(f"--- RESULTS ON RURAL DATA ---")
print(f"Naive Model MSE: {mse_naive:.4f}")
print(f"IPW-Corrected Model MSE: {mse_ipw:.4f}")
print(f"Error Reduction: {((mse_naive - mse_ipw) / mse_naive)*100:.2f}%")

--- RESULTS ON RURAL DATA ---
Naive Model MSE: 0.6117
IPW-Corrected Model MSE: 0.5675
Error Reduction: 7.22%
